### DATA READING

In [ ]:
dbutils.fs.ls("/Volumes/workspace/default/my_files/BigMart Sales.csv")

### DATA READING IN CSV FORMAT


In [ ]:
df = (
    spark.read.format("csv")
    .option("inferschema", True)
    .option("header", True)
    .load("/Volumes/workspace/default/my_files/BigMart Sales.csv")
)

In [ ]:
df.display()

### DATA READING IN JSON FORMAT

In [ ]:
df_json = (
    spark.read.format("json")
    .option("inferschema", True)
    .option("header", True)
    .option("multiline", False)
    .load("/Volumes/workspace/default/my_files/drivers.json")
)
df_json.display()

### DATA READING IN PARQUET FORMAT

In [ ]:
df_parquet = spark.read.format("parquet").load(
    "/Volumes/workspace/default/my_files/mtcars.parquet"
)

df_parquet.display()

In [ ]:
df_parquet1 = spark.read.format("parquet").load(
    "/Volumes/workspace/default/my_files/titanic.parquet"
)

df_parquet1.display()

### DATA READING IN AVRO FORMAT

In [ ]:
df_avro = spark.read.format("avro").load(
    "/Volumes/workspace/default/my_files/userdata5.avro"
)

df_avro.display()

###SCHEMA DEFINATION

In [ ]:
df.printSchema()

###DDL SCHEMA

In [ ]:
my_ddl_Schema = """
Item_Identifier string,
Item_Weight String,
Item_Fat_Content string,
Item_Visibility double,
Item_Type string,
Item_MRP double,
Outlet_Identifier string,
Outlet_Establishment_Year integer,
Outlet_Size string,
Outlet_Location_Type string,
Outlet_Type string,
Item_Outlet_Sales double
"""

In [ ]:
df = (
    spark.read.format("csv")
    .schema(my_ddl_Schema)
    .option("header", True)
    .load("/Volumes/workspace/default/my_files/BigMart Sales.csv")
)
df.printSchema()
df.display()

###STRUCT_TYPE SCHEMA

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [ ]:
my_struct_schema = StructType(
    [
        StructField("Item_Identifier", StringType(), True),
        StructField("Item_Weight", StringType(), True),
        StructField("Item_Fat_Content", StringType(), True),
        StructField("Item_Visibility", StringType(), True),
        StructField("Item_Type", StringType(), True),
        StructField("Item_MRP", StringType(), True),
        StructField("Outlet_Identifier", StringType(), True),
        StructField("Outlet_Establishment_Year", StringType(), True),
        StructField("Outlet_Size", StringType(), True),
        StructField("Outlet_Location_Type", StringType(), True),
        StructField("Outlet_Type", StringType(), True),
        StructField("Item_Outlet_Sales", StringType(), True),
    ]
)

In [ ]:
df = (
    spark.read.format("csv")
    .schema(my_struct_schema)
    .option("header", True)
    .load("/Volumes/workspace/default/my_files/BigMart Sales.csv")
)
df.printSchema()

###SELECT TRANSFORMATION

In [ ]:
df.display()

In [ ]:
df_cel = df.select("Item_Identifier", "Item_Weight", "Item_Fat_Content")

df_cel.display()

In [ ]:
df_cel = df.select(
    col("Item_Identifier"), col("Item_Weight"), col("Item_Fat_Content")
).display()

###ALIAS TRANSFORMATION

In [ ]:
df.select(col("Item_Identifier").alias("Item_ID")).display()

###FILTER/WHERE TRANSFORMATION
#####1. Filter the data with fat content = regular

In [ ]:
df.filter(col("Item_Fat_Content") == "Regular").display()

#####2. Slice the data with item type = soft drinks and weight < 10

In [ ]:
df.filter(
    (col("Item_Type") == "Soft Drinks") & (col("Item_Weight").cast("double") < 10)
).display()

#####3. Fetch the data with tier in (Tier1 or Tier2) and outlet size is Null

In [ ]:
df.filter(
    (col("Outlet_Size").isNull())
    & (col("Outlet_Location_Type").isin("Tier 1", "Tier 2"))
).display()

###WithColumnRenamed Transformation

In [ ]:
df.withColumnRenamed("Item_Weight", "item_wt").display()

###withColumn
#####1. creating a new column

In [ ]:
df = df.withColumn("flag", lit("new"))
display(df)

In [ ]:
df.withColumn("multiply", col("Item_Weight") * col("Item_MRP")).display()

#####2. modify the existing column

In [ ]:
df.withColumn(
    "Item_Fat_Content", regexp_replace(col("Item_Fat_Content"), "Low Fat", "LF")
).withColumn(
    "Item_Fat_Content", regexp_replace(col("Item_Fat_Content"), "Regular", "reg")
).display()

###Type Casting

In [ ]:
df.withColumn("Item_Weight", col("Item_Weight").cast("string")).display()

###SORT/ORDER BY

#####1. sorting in Descending order where the first weight will be the heighest

In [ ]:
df.sort(col("Item_Weight").desc()).display()

#####2. sorting in ascending order based on particular column

In [ ]:
df.sort(col("Item_Visibility").asc()).display()

#####3. sorting based on multiple columns

In [ ]:
df.sort(["Item_Weight", "Item_Visibility"], ascending=[0, 0]).display()

#####4. perform sorting in descending order in one column and ascending order in another column

In [ ]:
df.sort(["Item_Weight", "Item_Visibility"], ascending=[0, 1]).display()

###LIMIT

In [ ]:
df.limit(10).display()

#####DROP

#####Scenario - 1

In [ ]:
df.drop("Item_Visibility").display()

#####Scenario - 2

In [ ]:
df.drop("Item_Visibility", "Item_Type").display()

###DROP_DUPLICATES

#####Scenario - 1

In [ ]:
df.dropDuplicates().display()

#####scenario - 2 

In [ ]:
df.drop_duplicates(subset=["Item_Type"]).display()

In [ ]:
df.distinct().display()

###UNION and UNION BY NAME

#####Preparing Dataframes

In [ ]:
data1 = [("1", "kad"), ("2", "sid")]
schema1 = "id STRING, name STRING"

df1 = spark.createDataFrame(data1, schema1)

data2 = [("3", "rahul"), ("4", "jas")]
schema2 = "id STRING, name STRING"

df2 = spark.createDataFrame(data2, schema2)

In [ ]:
df1.display()

In [ ]:
df2.display()

###UNION

In [ ]:
df1.union(df2).display()

In [ ]:
data1 = [
    (
        "kad",
        "1",
    ),
    (
        "sid",
        "2",
    ),
]
schema1 = "name STRING, id STRING"

df1 = spark.createDataFrame(data1, schema1)

df1.display()

In [ ]:
df1.union(df2).display()

###UNION BY NAME

In [ ]:
df1.unionByName(df2).display()

###String Functions

#####Initcap()

In [ ]:
df.select(initcap("Item_Type")).display()

In [ ]:
df.select(lower("Item_Type")).display()

In [ ]:
df.select(upper("Item_Type")).display()

In [ ]:
df.select(upper("Item_Type").alias("upper_item_type")).display()

###Date Functions

#####current_date

In [ ]:
df = df.withColumn(
    "curr_date", current_date()
)  #####is used to create a column in the dataframe
df.display()

#####date_add()

In [ ]:
df = df.withColumn("week_after", date_add("curr_date", 7))
df.display()

#####date_sub()

In [ ]:
df = df.withColumn("week_before", date_sub("curr_date", 7))
df.display()

In [ ]:
df = df.withColumn("week_before", date_add("curr_date", -7))
df.display()

#####DateDIFF

In [ ]:
df = df.withColumn("date_diff", datediff("curr_date", "week_before"))
df.display()

#####Date_Format()

In [ ]:
df = df.withColumn("week_before", date_format("week_before", "dd-MM-yyyy"))
df.display()

###Handling Nulls

#####Dropping Nulls

In [ ]:
df.dropna("all").display()

In [ ]:
df.dropna("any").display()

In [ ]:
df.dropna(subset=["Outlet_Size"]).display()

#####Filling Nulls

In [ ]:
df.fillna("NotAvailable").display()

In [ ]:
df.fillna("NotAvailable", subset=["Outlet_Size"]).display()

###SPLIT and Indexing

#####SPLIT

In [ ]:
df.withColumn("Outlet_Type", split("Outlet_Type", " ")).display()

#####indexing

In [ ]:
df.withColumn("Outlet_Type", split("Outlet_Type", " ")[1]).display()

#####explode

In [ ]:
df_exp = df.withColumn("Outlet_Type", split("Outlet_Type", " "))
df_exp.display()

In [ ]:
df_exp.withColumn("Outlet_Type", explode("Outlet_Type")).display()

In [ ]:
df_exp.display()

###ARRAY_CONTAINS

In [ ]:
df_exp.withColumn("Type1_flag", array_contains("Outlet_Type", "Type1")).display()

###GROUP BY

#####Scenario - 1

In [ ]:
df.display()

In [ ]:
df.groupBy("Item_Type").agg(sum("Item_MRP")).display()

#####Scenario - 2

In [ ]:
df.groupBy("Item_Type").agg(avg("Item_MRP")).display()

#####Scenario - 3

In [ ]:
df.groupBy("Item_Type", "Outlet_Size").agg(sum("Item_MRP").alias("Total_MRP")).display()

#####Scenario - 4

In [ ]:
df.groupBy("Item_Type", "Outlet_Size").agg(sum("Item_MRP"), avg("Item_MRP")).display()

###Collect_list

In [ ]:
data = [
    ("user1", "book1"),
    ("user1", "book2"),
    ("user2", "book2"),
    ("user2", "book4"),
    ("user3", "book1"),
]

schema = "user string, book string"

df_book = spark.createDataFrame(data, schema)

df_book.display()

In [ ]:
df_book.groupBy("user").agg(collect_list("book")).display()

In [ ]:
df.select("Item_Type", "Outlet_Size", "Item_MRP").display()

###PIVOT

In [ ]:
df.groupBy("Item_Type").pivot("Outlet_Size").agg(avg("Item_MRP")).display()

###When_otherwise

#####Scenario - 1

In [ ]:
df = df.withColumn(
    "veg_exp_flag", when(col("Item_Type") == "Meat", "Non_veg").otherwise("veg")
)
display(df)

#####Scenario - 2

In [ ]:
df = df.withColumn(
    "veg_flag", when(col("Item_Type") == "Meat", "Non-Veg").otherwise("Veg")
)

In [ ]:
df.withColumn(
    "veg_exp_flag",
    when(((col("veg_flag") == "Veg") & (col("Item_MRP") < 100)), "Veg_Inexpensive")
    .when((col("veg_flag") == "Veg") & (col("Item_MRP") > 100), "Veg_Expensive")
    .otherwise("Non_Veg"),
).display()

###JOINS

In [ ]:
dataj1 = [
    ("1", "gaur", "d01"),
    ("2", "kit", "d02"),
    ("3", "sam", "d03"),
    ("4", "tim", "d03"),
    ("5", "aman", "d05"),
    ("6", "nad", "d06"),
]

schemaj1 = "emp_id STRING, emp_name STRING, dept_id STRING"

df1 = spark.createDataFrame(dataj1, schemaj1)

dataj2 = [
    ("d01", "HR"),
    ("d02", "Marketing"),
    ("d03", "Accounts"),
    ("d04", "IT"),
    ("d05", "Finance"),
]

schemaj2 = "dept_id STRING, department STRING"

df2 = spark.createDataFrame(dataj2, schemaj2)

In [ ]:
df1.display()

In [ ]:
df2.display()

#####Inner Join

In [ ]:
df1.join(df2, df1["dept_id"] == df2["dept_id"], "inner").display()

#####Left Join

In [ ]:
df1.join(df2, df1["dept_id"] == df2["dept_id"], "left").display()

#####Right Join

In [ ]:
df1.join(df2, df1["dept_id"] == df2["dept_id"], "right").display()

#####Anti Join

In [ ]:
df1.join(df2, df1["dept_id"] == df2["dept_id"], "anti").display()

###Window Functions

#####ROW_NUMBER()

In [ ]:
df.display()

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, col

In [ ]:
df.withColumn("rowcol", row_number().over(Window.orderBy("Item_Identifier"))).display()

#####RANK VS DENSE_RANK

In [ ]:
df.withColumn(
    "rank", rank().over(Window.orderBy(col("Item_Identifier").desc()))
).withColumn(
    "denseRank", dense_rank().over(Window.orderBy(col("Item_Identifier").desc()))
).display()

#####Cumilative Sum

In [ ]:
df.withColumn('cum_sum', sum('Item_MRP').over(Window.orderBy('Item_Type'))).display()

In [ ]:
df.withColumn('cum_sum', sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding, Window.currentRow))).display()


In [ ]:
df.withColumn('total_sum', sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing))).display()

###User Defined Functions

#####Step - 1

In [ ]:
def my_func(x):
    return x*x

#####Step - 2

In [ ]:
my_UDF = udf(my_func)

In [ ]:

df.withColumn('mynewcol',my_UDF('Item_MRP')).display()

###DATA WRITING

#####CSV

In [ ]:
df.write.format('csv').save('/Volumes/workspace/default/my_files/data.csv')

#####APPEND

In [ ]:
df.write.format('csv')\
    .mode('append')\
        .save('/Volumes/workspace/default/my_files/data.csv')

In [ ]:
df.write.format('csv')\
    .mode('append')\
    .option('path', '/Volumes/workspace/default/my_files/data.csv')\
    .save()

#####OVERWRITE

In [ ]:
df.write.format('csv')\
    .mode('overwrite')\
    .option('path', '/Volumes/workspace/default/my_files/data.csv')\
    .save()

#####ERROR

In [ ]:
df.write.format('csv')\
    .mode('error')\
    .option('path', '/Volumes/workspace/default/my_files/data.csv')\
    .save()

#####IGNORE

In [ ]:
df.write.format('csv')\
    .mode('ignore')\
    .option('path', '/Volumes/workspace/default/my_files/data.csv')\
    .save()

#####PARQUET

In [ ]:
df.write.format('parquet')\
    .mode('overwrite')\
    .option('path', '/Volumes/workspace/default/my_files/data.csv')\
    .save()

#####TABLE

In [ ]:
df.write.format('parquet')\
.mode('overwrite')\
.saveAsTable('my_table')

###SPARK SQL

#####CreateTempView

In [ ]:
df.createTempView('my_view')

In [ ]:
%sql
select * from my_view where Item_Fat_Content = 'Lf'

In [ ]:
df_sql = spark.sql("select * from my_view where Item_Fat_Content = 'Lf'")

In [ ]:
df_sql.display()